# 8. Results, the scarcity finding, and biosignature caution

The headline result, a full walkthrough of the current #1 candidate
combining every module in this notebook set, and why this project computes
no probability of life -- on principle, not as a placeholder.


This notebook is part of the reproducibility set for **Finding Earth 2.0 in
Distant Worlds**. It reads the same committed data every other output in this
project reads (`results/`, `data/processed/`, `data/manifests/`) and calls
the same `earth2` functions the pipeline itself calls -- nothing here is a
simplified restatement computed a different way. Run `python -m earth2 all`
first if `results/` does not exist yet.

See `docs/METHODS.md` for the full equations and `docs/LIMITATIONS.md` for
this project's stated caveats.


In [1]:
import sys
sys.path.insert(0, "../src")

import json
import pandas as pd

from earth2.config import RESULTS_DIR, WEB_DATA_DIR

summary = json.loads((RESULTS_DIR / "analysis_summary.json").read_text())
ranking = pd.read_parquet(RESULTS_DIR / "candidate_ranking.parquet")
planets = ranking[~ranking["is_control"].fillna(False)]


## The headline finding is a scarcity result

Of 6,354 confirmed planets, only a handful are simultaneously consistent
with the conservative habitable zone AND small enough to be plausibly rocky
-- and almost none of those have a mass that was actually measured rather
than predicted from the radius.


In [2]:
hz = summary["habitable_zone"]
n = summary["population"]["n_confirmed_planets"]
n_candidates = hz["n_conservative_hz_and_below_1p6_re"]
n_measured = hz["n_conservative_hz_and_below_1p6_re_with_measured_mass"]
print(f"Of {n:,} confirmed planets:")
print(f"  {hz['n_in_conservative_hz_nominal']:,} are in the conservative habitable zone")
print(f"  {n_candidates} are ALSO small enough to be plausibly rocky (R < 1.6 R_Earth)")
print(f"  {n_measured} of those has a dynamically measured mass")
print()
print("The search for Earth 2.0 is not currently limited by how many planets "
      "we know about. It is limited by how few of them we have measured well.")


Of 6,354 confirmed planets:
  174 are in the conservative habitable zone
  15 are ALSO small enough to be plausibly rocky (R < 1.6 R_Earth)
  1 of those has a dynamically measured mass

The search for Earth 2.0 is not currently limited by how many planets we know about. It is limited by how few of them we have measured well.


## The Gaia parallax cross-check, in one number

An independent validation this project did not have before the Gaia DR3
integration: how well does the archive's adopted distance agree with a
distance computed purely from Gaia's own parallax?


In [3]:
gaia = summary["gaia_crossmatch"]
print(f"{gaia['n_hosts_matched']:,} host systems cross-matched")
print(f"Median distance disagreement: {gaia['median_distance_disagreement_pct']}%")
print(f"{gaia['n_ruwe_above_1p4']} planets have a host with RUWE > 1.4 "
      "(possible unresolved binary -- context, not a score penalty)")


4,408 host systems cross-matched
Median distance disagreement: 0.733%
412 planets have a host with RUWE > 1.4 (possible unresolved binary -- context, not a score penalty)


## Case study: the current #1 candidate, every module together

The same deep-dive JSON the website's candidate page renders from --
Earth-2.0 index, Earth Similarity Index posterior, habitable-zone
probability, Gaia astrometric cross-check, and evidence coverage, for
whichever planet currently leads the ranking.


In [4]:
top = planets.sort_values("earth2_rank").iloc[0]
slug = top["pl_name"].replace(" ", "_").replace("/", "-")
dd = json.loads((WEB_DATA_DIR / "deepdive" / f"{slug}.json").read_text())

print(f"#{dd['ranking']['earth2_rank']}: {dd['planet']} ({dd['hostname']})")
print(f"  Earth-2.0 index: {dd['ranking']['earth2_index']:.3f}")
print(f"  Earth Similarity Index: {dd['earth_similarity']['esi_p50']:.3f} "
      f"({dd['earth_similarity']['esi_p16']:.3f}-{dd['earth_similarity']['esi_p84']:.3f}, 68% CI)")
print(f"  Conservative HZ probability: {dd['habitable_zone']['conservative_probability']:.1%}")
if dd.get("gaia_crossmatch"):
    g = dd["gaia_crossmatch"]
    print(f"  Gaia parallax distance: {g['distance_pc']} pc "
          f"({g['distance_disagreement_vs_archive_pct']}% vs. archive), RUWE={g['ruwe']}")
print(f"  Mass provenance: {dd['planet_parameters']['mass_earth']['class']}")


#1: Proxima Cen b (Proxima Cen)
  Earth-2.0 index: 0.876
  Earth Similarity Index: 0.915 (0.897-0.932, 68% CI)
  Conservative HZ probability: 100.0%
  Gaia parallax distance: 1.302 pc (0.06% vs. archive), RUWE=0.971
  Mass provenance: msini_lower_limit


## Biosignature caution: this project computes no probability of life

There is no calibrated likelihood function for biology on exoplanets --
one inhabited world, no confirmed uninhabited control with a comparable
atmosphere, and no complete theory of abiotic false positives. A number
like "68% chance of life" would be a fabricated statistic wearing a decimal
point. `earth2.spectroscopy.biosignature` instead returns interpretive
context: what a species would mean, what produces it without life, and
what would have to be true for a biological interpretation to survive
scrutiny -- never a verdict.


In [5]:
from earth2.spectroscopy.biosignature import biosignature_context_for

for species in ("O2", "CH4", "CO2"):
    ctx = biosignature_context_for(species)["context"]
    print(f"{species}:")
    print(f"  Why discussed:     {ctx['why_discussed']}")
    print(f"  Why not conclusive: {ctx['why_not_conclusive']}")
    print()


O2:
  Why discussed:     Earth's atmospheric oxygen is overwhelmingly biological in origin.
  Why not conclusive: Several well-studied abiotic routes can produce comparable abundances, particularly around M dwarfs, which host most of the small planets we can currently characterise.

CH4:
  Why discussed:     Biologically produced on Earth and short-lived, so it must be replenished.
  Why not conclusive: Serpentinisation and volcanism produce it without biology.

CO2:
  Why discussed:     Establishes that an atmosphere exists and constrains its composition.
  Why not conclusive: Dominant on Venus and Mars, neither of which is inhabited at the surface.



## What this project is, and is not

An Earth-2.0 index ranks physical **similarity** and **observational
evidence** -- not the probability that a planet supports life. Every number
in this notebook set was computed by the pipeline itself, from real public
archive data, with every assumption and every known limitation stated
alongside the result rather than left for a reader to discover on their own.
See `docs/LIMITATIONS.md` for the complete list.
